# AlphaLens Model Registry

This notebook audits the latest immutable model package. Registration stores the native model, ordered feature contract, data fingerprints, metrics, model card, reference predictions, and artifact checksums. Promotion is automatic and evidence-based; a rejected research model remains reproducible but cannot be served as champion.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipelines.ml.registry import load_registered_model, verify_registered_model

REGISTRY = ROOT / 'data' / 'ml' / 'registry'

## Registry index

In [ ]:
index_path = REGISTRY / 'registry.json'
if not index_path.exists():
    raise FileNotFoundError('Run python -m pipelines.ml.registry first.')
index = json.loads(index_path.read_text(encoding='utf-8'))
display(pd.DataFrame(index['models']))
print('Latest:', index['latest_version'])
print('Champion:', index['champion_version'])

## Integrity and reproducibility verification

In [ ]:
version = index['latest_version']
version_directory = REGISTRY / version
manifest = json.loads((version_directory / 'manifest.json').read_text(encoding='utf-8'))
verification = verify_registered_model(version_directory)
display(pd.Series(verification, name='value'))
assert verification['passed']

In [ ]:
artifacts = pd.DataFrame([
    {'artifact': name, **metadata}
    for name, metadata in manifest['artifacts'].items()
]).sort_values('artifact')
display(artifacts)

## Promotion decision

In [ ]:
promotion = manifest['promotion']
display(pd.DataFrame([
    {'gate': name, **details}
    for name, details in promotion['checks'].items()
]))
print('Status:', promotion['status'].upper())
for reason in promotion['reasons']:
    print('-', reason)

## Feature and data contract

In [ ]:
schema = json.loads((version_directory / 'feature_schema.json').read_text(encoding='utf-8'))
display(pd.DataFrame(schema['features']))
display(pd.Series(manifest['dataset_fingerprints'], name='sha256'))
display(pd.Series(manifest['training_window'], name='value'))

## Policy-aware loading

A rejected model requires an explicit research override. Production inference will request `champion` and therefore fail closed when no model passes the gates.

In [ ]:
registered = load_registered_model(REGISTRY, version='latest', allow_rejected=True)
print('Loaded for research:', registered.version)
print('Boosted rounds:', registered.model.get_booster().num_boosted_rounds())
print('Features:', registered.model.get_booster().num_features())

In [ ]:
display(Markdown((version_directory / 'model_card.md').read_text(encoding='utf-8')))